# 100 — Computer Use Agent
## Build a headless bash + file-editor action loop powered by Anthropic's computer-use beta
⏱ ~60 min

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Esturban/agent/blob/master/examples/100-computer-use-agent/computer_use_agent_workbook.ipynb)

Traditional software automates computers by hardcoding every step. **Computer use agents** flip this: you describe *what* you want, and the model decides *how* to do it — choosing tools, writing files, running commands, and reading outputs in a self-directed loop.

Anthropic's `computer-use-2024-10-22` beta unlocks three built-in tools:
- `bash_20241022` — execute shell commands
- `text_editor_20241022` — create, view, and patch text files
- `computer_20241022` — mouse and keyboard control (requires a display / Xvfb)

This workshop focuses on the **headless subset** (bash + editor), which runs anywhere without a GUI — a perfect introduction to the action-loop pattern before moving to full desktop automation.

---
### Workshop Roadmap
| # | Topic |
|---|-------|
| 1 | **Concepts** — what computer use is and why it matters |
| 2 | **Setup** — install deps, configure API key |
| 3 | **Tool Definitions** — bash and text_editor tool specs |
| 4 | **Tool Executors** — local Python code that runs each tool |
| 5 | **The Action Loop** — the core `run_agent` function |
| 6 | **Live Demo** — write + run a Fibonacci script end-to-end |
| 7 | **Anatomy of a Response** — reading `ToolUseBlock` objects |
| 8 | **Extending the Agent** — adding tools, custom tasks |
| ★ | **Exercises + Answer Key** |

---
### Prerequisites
- Python 3.10+, or Google Colab
- `ANTHROPIC_API_KEY` in `.env` or Colab Secrets
- `anthropic`, `python-dotenv` packages

### Key References
> Anthropic (2024). *Computer use (beta)* — https://docs.anthropic.com/en/docs/agents-and-tools/computer-use  
> Model card: `claude-3-5-sonnet-20241022` — the first model released with computer-use capability

In [ ]:
import sys

def _in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

if _in_colab():
    import subprocess
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "anthropic", "python-dotenv"],
        check=True
    )
    print("Colab install complete.")
else:
    print("Local env — skipping install (ensure anthropic and python-dotenv are installed).")

In [ ]:
import os

try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
    print("Loaded ANTHROPIC_API_KEY from Colab Secrets.")
except ImportError:
    from dotenv import load_dotenv
    load_dotenv()
    print("Loaded from .env file.")

key = os.environ.get("ANTHROPIC_API_KEY", "")
print(f"API key ready: {bool(key) and key.startswith('sk-')}")

---
## Part 1 — Concepts: What Is Computer Use?

### The Old Way: Procedural Automation

Traditional automation tools (Selenium, PyAutoGUI, shell scripts) require you to specify *every step explicitly*:

```
1. Open file X
2. Write exactly this content
3. Run exactly this command
4. Parse output with this regex
```

This breaks the moment anything changes — a UI update, a different file path, a new error message.

### The Computer Use Way: Goal-Directed Automation

Computer use agents receive a **goal** and autonomously decide how to achieve it:

```
Goal: "Write a Fibonacci script and run it"

Model decides:
  Step 1 → use text_editor to create /tmp/fib.py
  Step 2 → use bash to run python /tmp/fib.py
  Step 3 → read output, verify result, signal end_turn
```

### The Action Loop Pattern

Every computer use agent follows this cycle:

```
┌─────────────────────────────────────────────┐
│                                             │
│   User task ──► Model API call              │
│                     │                       │
│            ┌────────▼────────┐              │
│            │  ToolUseBlocks  │              │
│            └────────┬────────┘              │
│                     │                       │
│            ┌────────▼────────┐              │
│            │ Execute locally │              │
│            │ (bash / editor) │              │
│            └────────┬────────┘              │
│                     │                       │
│            ┌────────▼────────┐              │
│            │  tool_result    │              │
│            │  back to model  │◄─────────┐  │
│            └────────┬────────┘          │  │
│                     │                   │  │
│            stop_reason == end_turn?     │  │
│                  YES ──► Done           │  │
│                  NO  ──────────────────►┘  │
│                                             │
└─────────────────────────────────────────────┘
```

### Why `betas=["computer-use-2024-10-22"]`?

These tools are still in beta. The beta flag:
- Unlocks the specialized tool types (`bash_20241022`, `text_editor_20241022`, `computer_20241022`)
- Applies Anthropic's computer-use safety filters
- Signals your acceptance of beta terms

Without the flag, the API rejects those tool type strings.

---
## Part 2 — Setup: Model and Tool Definitions

The agent's configuration lives in `src/tools.py`. Let's define it directly here so the notebook is self-contained.

Key decisions:
- **`MODEL`**: `claude-3-5-sonnet-20241022` is the first release with computer-use capability
- **`MAX_STEPS`**: A safety ceiling — prevents infinite loops if the model gets stuck
- **`CU_TOOLS`**: The tool specs sent to the API. The `type` field uses the beta versioned name.

Notice that `bash_20241022` takes a `command` input, while `text_editor_20241022` takes structured params (`command`, `path`, `file_text`, etc.).

In [ ]:
# ── Configuration (mirrors src/tools.py) ───────────────────────────────────

MODEL = "claude-3-5-sonnet-20241022"
MAX_STEPS = 10

# bash + text_editor run without a display.
# Add "computer_20241022" + display config for mouse/keyboard control.
CU_TOOLS = [
    {"type": "bash_20241022",        "name": "bash"},
    {"type": "text_editor_20241022", "name": "str_replace_editor"},
]

TASK = (
    "Write a Python script called /tmp/fib.py that prints the first 10 Fibonacci numbers, "
    "one per line. Run it and show me the output."
)

print("Model:", MODEL)
print("Max steps:", MAX_STEPS)
print("Tools registered:", [t["name"] for t in CU_TOOLS])
print("\nTask:")
print(" ", TASK)

### Tool Spec Deep Dive

Unlike normal function-calling tools (where you define the schema yourself), computer-use tools use **pre-defined schemas** baked into the beta. You only supply `type` and `name`:

| Tool type | Name (your choice) | What the model sends as `input` |
|-----------|-------------------|---------------------------------|
| `bash_20241022` | `bash` | `{"command": "python /tmp/fib.py"}` |
| `text_editor_20241022` | `str_replace_editor` | `{"command": "create", "path": "/tmp/fib.py", "file_text": "..."}` |
| `computer_20241022` | `computer` | `{"action": "screenshot"}` or `{"action": "left_click", "coordinate": [x, y]}` |

The `text_editor_20241022` supports four commands:
- `create` — write a new file
- `view` — read a file
- `str_replace` — find and replace text
- `insert` — insert text at a line number

---
## Part 3 — Tool Executors

When the model requests a tool call, *your code* is responsible for executing it and returning results. The model has no direct access to your system — it's just generating structured JSON describing what it wants done.

We need two executor functions:
1. `_run_bash(command)` — runs a shell command via `subprocess`
2. `_run_editor(params)` — implements the four text_editor commands using Python's file I/O

These are the **trust boundary**: you decide what's allowed. A production agent would add sandboxing, path restrictions, and command allowlists here.

In [ ]:
import subprocess

def _run_bash(command: str) -> str:
    """Execute a shell command; return stdout or a formatted error."""
    try:
        r = subprocess.run(
            command, shell=True,
            capture_output=True, text=True, timeout=15
        )
        return r.stdout or (
            f"[exit {r.returncode}] {r.stderr.strip()}"
            if r.returncode else "(no output)"
        )
    except Exception as e:
        return f"Error: {e}"


def _run_editor(params: dict) -> str:
    """Minimal text_editor_20241022 executor: create, view, str_replace."""
    cmd  = params.get("command", "view")
    path = params.get("path", "")

    if cmd == "create":
        with open(path, "w") as f:
            f.write(params.get("file_text", ""))
        return f"Created {path}"

    if cmd == "view":
        try:
            return open(path).read()
        except FileNotFoundError:
            return f"Not found: {path}"

    if cmd == "str_replace":
        content = open(path).read()
        open(path, "w").write(
            content.replace(params.get("old_str", ""), params.get("new_str", ""))
        )
        return "Done"

    return "Unknown editor command"


# Quick sanity check — no API needed
print(_run_bash("echo 'bash executor works'"))
print(_run_editor({"command": "create", "path": "/tmp/test_wb.txt", "file_text": "hello notebook\n"}))
print(_run_editor({"command": "view",   "path": "/tmp/test_wb.txt"}))

### Why `subprocess` with `shell=True`?

The model sends commands like `"python /tmp/fib.py"` or `"ls -la /tmp"` as plain strings. Using `shell=True` lets us pass these directly to `bash` without tokenising them.

**Security note**: `shell=True` is dangerous with untrusted input — it allows shell injection. In production:
- Use `shell=False` with a tokenised argument list
- Run in a Docker container or VM sandbox
- Allowlist commands at the executor level
- Set `timeout` (already done here) to prevent runaway processes

For this workshop, we trust the model's output since we control the task.

---
## Part 4 — The Action Loop

This is the heart of the agent. The `run_agent` function:

1. Sends the task as a user message
2. Receives a response containing `ToolUseBlock` objects
3. Executes each requested tool locally
4. Appends `tool_result` messages back to the conversation
5. Repeats until `stop_reason == "end_turn"` or no tool calls remain

### The message thread structure

```
messages = [
  {"role": "user",      "content": "Write fib.py and run it"},
  {"role": "assistant", "content": [ToolUseBlock(id='tu_1', name='str_replace_editor', input={...})]},
  {"role": "user",      "content": [{"type": "tool_result", "tool_use_id": "tu_1", "content": "Created /tmp/fib.py"}]},
  {"role": "assistant", "content": [ToolUseBlock(id='tu_2', name='bash', input={"command": "python /tmp/fib.py"})]},
  {"role": "user",      "content": [{"type": "tool_result", "tool_use_id": "tu_2", "content": "0\n1\n1\n..."}]},
  {"role": "assistant", "content": [TextBlock(text="Here are the first 10 Fibonacci numbers...")]},
]
```

Each `tool_result` must reference the exact `tool_use_id` from the model's request — this is how the model matches results to its requests when it issues multiple tool calls in one turn.

In [ ]:
import anthropic

def run_agent(task: str) -> list[dict]:
    """Action loop: send task → receive ToolUseBlocks → execute → feed results → repeat.

    Each iteration the model sees previous tool outputs and decides the next action.
    Stops on stop_reason='end_turn' or when no tool calls remain.
    """
    client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    messages = [{"role": "user", "content": task}]
    steps: list[dict] = []

    for step_num in range(MAX_STEPS):
        print(f"  [loop] Iteration {step_num + 1} — calling API...")

        resp = client.beta.messages.create(
            model=MODEL,
            max_tokens=4096,
            tools=CU_TOOLS,
            messages=messages,
            betas=["computer-use-2024-10-22"],
        )

        # Collect tool use blocks and any narration text
        tool_uses = [b for b in resp.content if b.type == "tool_use"]
        for b in resp.content:
            if b.type == "text" and b.text.strip():
                steps.append({"type": "text", "content": b.text.strip()})

        print(f"  [loop] stop_reason={resp.stop_reason!r}, tool_calls={len(tool_uses)}")

        # Exit conditions
        if resp.stop_reason == "end_turn" or not tool_uses:
            break

        # Execute each tool call and gather results
        results = []
        for tu in tool_uses:
            if tu.name == "bash":
                output = _run_bash(tu.input.get("command", ""))
            else:
                output = _run_editor(tu.input)

            steps.append({
                "type":   "action",
                "tool":   tu.name,
                "input":  tu.input,
                "output": output,
            })
            results.append({
                "type":        "tool_result",
                "tool_use_id": tu.id,
                "content":     output,
            })

        # Extend the conversation thread
        messages += [
            {"role": "assistant", "content": resp.content},
            {"role": "user",      "content": results},
        ]

    return steps

print("run_agent defined — ready to run.")

---
## Part 5 — Live Demo: Fibonacci End-to-End

Now we run the full agent. Watch the loop iterations in the output — each one is a round-trip to the Anthropic API.

Expected behaviour:
1. Model calls `str_replace_editor` with `command=create` to write `/tmp/fib.py`
2. Model calls `bash` with `command=python /tmp/fib.py`
3. Model receives the Fibonacci output, narrates the result, signals `end_turn`

**This cell makes real API calls and will incur charges.**

In [ ]:
print("Task:", TASK)
print("=" * 60)

steps = run_agent(TASK)

print("\n" + "=" * 60)
print("STEP LOG")
print("=" * 60)

for i, step in enumerate(steps, 1):
    if step["type"] == "text":
        print(f"\n[{i}] model narration:")
        print(f"    {step['content']}")
    else:
        print(f"\n[{i}] tool={step['tool']}")
        print(f"    input:  {step['input']}")
        print(f"    output: {step['output'][:300]}")

print("\n" + "=" * 60)
print(f"Done — {len(steps)} steps recorded.")

---
## Part 6 — Anatomy of a Response

Let's look at what the Anthropic API actually returns. A `beta.messages.create` response has:

```python
resp.id            # message ID
resp.model         # which model responded
resp.stop_reason   # "tool_use" | "end_turn" | "max_tokens"
resp.content       # list of content blocks
resp.usage         # input_tokens, output_tokens
```

### Content block types

| Block type | Class | Key fields |
|------------|-------|------------|
| `text` | `TextBlock` | `.text` |
| `tool_use` | `ToolUseBlock` | `.id`, `.name`, `.input` |

### `stop_reason` values

| Value | Meaning |
|-------|---------|
| `"tool_use"` | Model wants to call tools — keep looping |
| `"end_turn"` | Model is done — break the loop |
| `"max_tokens"` | Hit token limit — handle gracefully |

The `stop_reason` is the **primary loop control signal**. Always check it before deciding whether to continue.

In [ ]:
# Inspect one raw API response to see the block structure
import json

client = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

probe_resp = client.beta.messages.create(
    model=MODEL,
    max_tokens=512,
    tools=CU_TOOLS,
    messages=[{"role": "user", "content": "List the files in /tmp — use bash."}],
    betas=["computer-use-2024-10-22"],
)

print("stop_reason:", probe_resp.stop_reason)
print("usage:", probe_resp.usage)
print(f"\ncontent blocks ({len(probe_resp.content)}):")
for block in probe_resp.content:
    print(f"  type={block.type!r}")
    if block.type == "tool_use":
        print(f"    id   = {block.id}")
        print(f"    name = {block.name}")
        print(f"    input= {block.input}")

---
## Part 7 — Step Trace Analysis

Let's write a helper to pretty-print the step log and understand the agent's decision-making process. Tracing steps is essential for debugging real agents.

In [ ]:
def print_trace(steps: list[dict]) -> None:
    """Pretty-print the step log with type-based formatting."""
    action_steps  = [s for s in steps if s["type"] == "action"]
    narrate_steps = [s for s in steps if s["type"] == "text"]
    tools_used    = [s["tool"] for s in action_steps]

    print(f"Total steps    : {len(steps)}")
    print(f"Tool calls     : {len(action_steps)}")
    print(f"Narrations     : {len(narrate_steps)}")
    print(f"Tools invoked  : {tools_used}")
    print()

    for i, step in enumerate(steps, 1):
        if step["type"] == "text":
            banner = f"── [{i}] MODEL NARRATION "
            print(banner + "─" * max(0, 60 - len(banner)))
            print(step["content"])
        else:
            banner = f"── [{i}] TOOL: {step['tool'].upper()} "
            print(banner + "─" * max(0, 60 - len(banner)))
            print("  INPUT :", json.dumps(step["input"], indent=4))
            print("  OUTPUT:", step["output"][:400])
        print()

# Re-use steps from Part 5
print_trace(steps)

---
## Part 8 — Extending the Agent

### Changing the task

The simplest extension is a different task. The agent will plan its own steps — you don't change the loop.

### Adding a custom tool

You can mix computer-use tools with regular function-calling tools. For example, adding a `read_url` tool that fetches web content:

```python
EXTENDED_TOOLS = CU_TOOLS + [
    {
        "name": "read_url",
        "description": "Fetch the text content of a URL.",
        "input_schema": {
            "type": "object",
            "properties": {"url": {"type": "string"}},
            "required": ["url"]
        }
    }
]
```

You'd then add an `elif tu.name == "read_url"` branch in the executor.

### Safety considerations

As you expand tool access, consider:

| Risk | Mitigation |
|------|------------|
| Prompt injection via file content | Sanitize file reads before feeding back |
| Accidental deletion | Add path restrictions in `_run_editor` |
| Infinite loops | `MAX_STEPS` ceiling (already in place) |
| Network exfiltration | Disable network in sandbox for sensitive tasks |
| Resource exhaustion | `timeout=15` in `subprocess.run` (already done) |

In [ ]:
# Try a different task — the loop is identical, only the goal changes
TASK_2 = (
    "Write a Python script at /tmp/primes.py that prints all prime numbers up to 50. "
    "Run it with bash and show me the output."
)

print("Task:", TASK_2)
print("=" * 60)

steps_2 = run_agent(TASK_2)

print("\n" + "=" * 60)
print_trace(steps_2)

---
## Part 9 — Building a Reusable Agent Class

Wrapping `run_agent` in a class makes it easier to configure different agent instances with different tool sets, models, and task constraints.

In [ ]:
class ComputerUseAgent:
    """Configurable wrapper around the computer-use action loop."""

    def __init__(
        self,
        model: str = MODEL,
        max_steps: int = MAX_STEPS,
        tools: list | None = None,
    ):
        self.model     = model
        self.max_steps = max_steps
        self.tools     = tools or CU_TOOLS
        self.client    = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])

    def _execute(self, tu) -> str:
        """Route a ToolUseBlock to the right executor."""
        if tu.name == "bash":
            return _run_bash(tu.input.get("command", ""))
        return _run_editor(tu.input)

    def run(self, task: str) -> list[dict]:
        """Execute the action loop and return the step log."""
        messages = [{"role": "user", "content": task}]
        steps: list[dict] = []

        for _ in range(self.max_steps):
            resp = self.client.beta.messages.create(
                model=self.model,
                max_tokens=4096,
                tools=self.tools,
                messages=messages,
                betas=["computer-use-2024-10-22"],
            )

            tool_uses = [b for b in resp.content if b.type == "tool_use"]
            for b in resp.content:
                if b.type == "text" and b.text.strip():
                    steps.append({"type": "text", "content": b.text.strip()})

            if resp.stop_reason == "end_turn" or not tool_uses:
                break

            results = []
            for tu in tool_uses:
                output = self._execute(tu)
                steps.append({"type": "action", "tool": tu.name,
                               "input": tu.input, "output": output})
                results.append({"type": "tool_result",
                                 "tool_use_id": tu.id, "content": output})

            messages += [
                {"role": "assistant", "content": resp.content},
                {"role": "user",      "content": results},
            ]

        return steps

    def summary(self, steps: list[dict]) -> str:
        """Return a one-line summary of the completed run."""
        actions  = sum(1 for s in steps if s["type"] == "action")
        narrates = sum(1 for s in steps if s["type"] == "text")
        tools    = [s["tool"] for s in steps if s["type"] == "action"]
        return f"{len(steps)} steps: {actions} tool calls ({tools}), {narrates} narrations"


# Verify the class works
agent = ComputerUseAgent()
print("Agent created — model:", agent.model)

In [ ]:
# Run a third task using the class interface
TASK_3 = (
    "Write /tmp/hello.py that prints 'Hello from computer-use!' and run it."
)

agent = ComputerUseAgent(max_steps=5)
steps_3 = agent.run(TASK_3)
print(agent.summary(steps_3))
print_trace(steps_3)

---
## Part 10 — Token Tracking and Cost Awareness

Each API call consumes tokens. In a multi-step loop, tokens accumulate because the full conversation history is sent every iteration. Understanding this helps you set `MAX_STEPS` appropriately and estimate costs.

**Pricing reference (as of October 2024):**
- Input tokens: $3 / 1M
- Output tokens: $15 / 1M

The conversation grows linearly with steps — a 10-step agent might use 5-10x the tokens of a 1-step agent due to the growing message history.

In [ ]:
def run_agent_with_costs(task: str) -> tuple[list[dict], dict]:
    """Like run_agent but also tracks cumulative token usage."""
    client   = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    messages = [{"role": "user", "content": task}]
    steps: list[dict] = []
    usage   = {"input_tokens": 0, "output_tokens": 0, "api_calls": 0}

    for _ in range(MAX_STEPS):
        resp = client.beta.messages.create(
            model=MODEL, max_tokens=4096,
            tools=CU_TOOLS, messages=messages,
            betas=["computer-use-2024-10-22"],
        )

        usage["input_tokens"]  += resp.usage.input_tokens
        usage["output_tokens"] += resp.usage.output_tokens
        usage["api_calls"]     += 1

        tool_uses = [b for b in resp.content if b.type == "tool_use"]
        for b in resp.content:
            if b.type == "text" and b.text.strip():
                steps.append({"type": "text", "content": b.text.strip()})

        if resp.stop_reason == "end_turn" or not tool_uses:
            break

        results = []
        for tu in tool_uses:
            output = _run_bash(tu.input.get("command", "")) if tu.name == "bash" else _run_editor(tu.input)
            steps.append({"type": "action", "tool": tu.name, "input": tu.input, "output": output})
            results.append({"type": "tool_result", "tool_use_id": tu.id, "content": output})

        messages += [
            {"role": "assistant", "content": resp.content},
            {"role": "user",      "content": results},
        ]

    # Approximate USD cost
    usage["estimated_usd"] = (
        usage["input_tokens"]  / 1_000_000 * 3 +
        usage["output_tokens"] / 1_000_000 * 15
    )
    return steps, usage


steps_4, usage_4 = run_agent_with_costs(
    "Write /tmp/greet.py that accepts a name arg and prints 'Hello, <name>!'. Run it with 'World'."
)

print(f"API calls     : {usage_4['api_calls']}")
print(f"Input tokens  : {usage_4['input_tokens']:,}")
print(f"Output tokens : {usage_4['output_tokens']:,}")
print(f"Estimated cost: ${usage_4['estimated_usd']:.4f}")

---
## Part 11 — Adding a System Prompt

You can guide the agent's behaviour with a `system` prompt. This is useful for:
- Restricting what directories/commands are allowed
- Setting a persona or output format
- Pre-loading context (project structure, conventions)

The `system` parameter is passed alongside `messages` in the API call.

In [ ]:
SYSTEM_PROMPT = """\
You are a careful Python developer. When writing scripts:
- Always include a docstring at the top of the file
- Use type annotations on all functions
- After running a script, summarise the output in one sentence
Only write files to /tmp/. Never access any other directory.
"""

def run_agent_with_system(task: str, system: str) -> list[dict]:
    """Run the action loop with a system prompt."""
    client   = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    messages = [{"role": "user", "content": task}]
    steps: list[dict] = []

    for _ in range(MAX_STEPS):
        resp = client.beta.messages.create(
            model=MODEL, max_tokens=4096,
            tools=CU_TOOLS, messages=messages,
            system=system,
            betas=["computer-use-2024-10-22"],
        )

        tool_uses = [b for b in resp.content if b.type == "tool_use"]
        for b in resp.content:
            if b.type == "text" and b.text.strip():
                steps.append({"type": "text", "content": b.text.strip()})

        if resp.stop_reason == "end_turn" or not tool_uses:
            break

        results = []
        for tu in tool_uses:
            output = _run_bash(tu.input.get("command", "")) if tu.name == "bash" else _run_editor(tu.input)
            steps.append({"type": "action", "tool": tu.name, "input": tu.input, "output": output})
            results.append({"type": "tool_result", "tool_use_id": tu.id, "content": output})

        messages += [
            {"role": "assistant", "content": resp.content},
            {"role": "user",      "content": results},
        ]

    return steps


steps_sys = run_agent_with_system(
    task="Write /tmp/squares.py that prints the squares of 1 to 5 and run it.",
    system=SYSTEM_PROMPT
)
print_trace(steps_sys)

---
## Part 12 — What About `computer_20241022`?

The third tool — `computer_20241022` — enables mouse and keyboard control. It requires:

1. A running display (X11 virtual framebuffer via Xvfb, or a real display)
2. Screenshot capability (the model requests screenshots to see the screen state)
3. A screenshot executor that returns base64-encoded images

**Why we skipped it**: Colab and most local environments don't have a display by default.

**To add it in a Docker/Xvfb environment:**

```python
import pyautogui, base64
from PIL import ImageGrab
import io

CU_TOOLS_FULL = [
    {"type": "bash_20241022",        "name": "bash"},
    {"type": "text_editor_20241022", "name": "str_replace_editor"},
    {"type": "computer_20241022",    "name": "computer",
     "display_width_px": 1920, "display_height_px": 1080,
     "display_number": 1},
]

def _run_computer(params: dict) -> str | dict:
    action = params.get("action")
    if action == "screenshot":
        img = ImageGrab.grab()
        buf = io.BytesIO()
        img.save(buf, format="PNG")
        b64 = base64.b64encode(buf.getvalue()).decode()
        return {"type": "image", "source": {"type": "base64",
                "media_type": "image/png", "data": b64}}
    if action == "left_click":
        x, y = params["coordinate"]
        pyautogui.click(x, y)
        return f"Clicked ({x}, {y})"
    if action == "type":
        pyautogui.typewrite(params["text"])
        return f"Typed: {params['text']}"
    return f"Unsupported action: {action}"
```

Then add `elif tu.name == "computer": output = _run_computer(tu.input)` to your executor dispatch.

The model would then alternate `screenshot → analyse → click/type → screenshot → ...` to navigate GUIs visually.

In [ ]:
# Demonstration: what the computer tool spec looks like (no API call)
COMPUTER_TOOL_SPEC = {
    "type": "computer_20241022",
    "name": "computer",
    "display_width_px": 1920,
    "display_height_px": 1080,
    "display_number": 1,  # matches :1 in Xvfb
}

FULL_TOOL_SUITE = [
    {"type": "bash_20241022",        "name": "bash"},
    {"type": "text_editor_20241022", "name": "str_replace_editor"},
    COMPUTER_TOOL_SPEC,
]

print("Full tool suite for GUI automation:")
for t in FULL_TOOL_SUITE:
    print(f"  {t['type']:35s} → name='{t['name']}'")

print("\nNote: computer_20241022 requires a display. Set DISPLAY=:1 and start Xvfb.")

---
## Exercises

### Exercise 1 — Count the vowels
Write a new task string `TASK_EX1` that asks the agent to:
1. Write `/tmp/vowels.py` — a script that counts vowels in the string `"Hello World from Claude"`
2. Run it and show the count

Run it with `run_agent(TASK_EX1)` and print the step trace.

---

### Exercise 2 — Add a timeout guard to `_run_bash`
The current `_run_bash` has `timeout=15`. Extend it to:
- Accept a `timeout` parameter (default 15)
- Return a specific `"[TIMEOUT]"` string when `subprocess.TimeoutExpired` is raised
- Test it with `_run_bash("sleep 20", timeout=2)` — it should return `"[TIMEOUT]"`

---

### Exercise 3 — Max steps early exit
Create a version of `run_agent` called `run_agent_limited` that:
- Accepts a `max_steps` parameter
- Appends a `{"type": "limit_reached"}` entry to `steps` if the loop exits due to hitting the limit
- Returns both `steps` and a bool `hit_limit`

Test with `max_steps=1` on the Fibonacci task — it should hit the limit and return `hit_limit=True`.

In [ ]:
# ── Exercise 1 Answer Key ────────────────────────────────────────────────────

TASK_EX1 = (
    "Write a Python script at /tmp/vowels.py that counts the vowels "
    "in the string 'Hello World from Claude' and prints the count. "
    "Run it with bash."
)

steps_ex1 = run_agent(TASK_EX1)
print_trace(steps_ex1)

In [ ]:
# ── Exercise 2 Answer Key ────────────────────────────────────────────────────

def _run_bash_with_timeout(command: str, timeout: int = 15) -> str:
    """Execute a shell command with a configurable timeout."""
    try:
        r = subprocess.run(
            command, shell=True,
            capture_output=True, text=True,
            timeout=timeout
        )
        return r.stdout or (
            f"[exit {r.returncode}] {r.stderr.strip()}"
            if r.returncode else "(no output)"
        )
    except subprocess.TimeoutExpired:
        return "[TIMEOUT]"
    except Exception as e:
        return f"Error: {e}"


# Quick test — should complete normally
print("Normal:",  _run_bash_with_timeout("echo hello", timeout=5))

# Should timeout (adjust if sleep 20 is too slow on your machine)
print("Timeout:", _run_bash_with_timeout("sleep 20", timeout=2))

In [ ]:
# ── Exercise 3 Answer Key ────────────────────────────────────────────────────

def run_agent_limited(task: str, max_steps: int = MAX_STEPS) -> tuple[list[dict], bool]:
    """Run the action loop with a configurable step limit.

    Returns (steps, hit_limit) where hit_limit is True if MAX_STEPS was reached.
    """
    client   = anthropic.Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    messages = [{"role": "user", "content": task}]
    steps: list[dict] = []
    hit_limit = False

    for step_num in range(max_steps):
        resp = client.beta.messages.create(
            model=MODEL, max_tokens=4096,
            tools=CU_TOOLS, messages=messages,
            betas=["computer-use-2024-10-22"],
        )

        tool_uses = [b for b in resp.content if b.type == "tool_use"]
        for b in resp.content:
            if b.type == "text" and b.text.strip():
                steps.append({"type": "text", "content": b.text.strip()})

        if resp.stop_reason == "end_turn" or not tool_uses:
            break

        results = []
        for tu in tool_uses:
            output = _run_bash(tu.input.get("command", "")) if tu.name == "bash" else _run_editor(tu.input)
            steps.append({"type": "action", "tool": tu.name, "input": tu.input, "output": output})
            results.append({"type": "tool_result", "tool_use_id": tu.id, "content": output})

        messages += [
            {"role": "assistant", "content": resp.content},
            {"role": "user",      "content": results},
        ]

        # Check if this was the last allowed iteration
        if step_num == max_steps - 1 and resp.stop_reason != "end_turn":
            hit_limit = True
            steps.append({"type": "limit_reached"})

    return steps, hit_limit


# Test: max_steps=1 should hit the limit on the Fibonacci task
steps_lim, did_hit = run_agent_limited(TASK, max_steps=1)
print("hit_limit:", did_hit)
print("final step:", steps_lim[-1] if steps_lim else "(empty)")

---
## Workshop Complete

You have built a complete **Anthropic computer-use agent** from scratch:

| Component | What you learned |
|-----------|------------------|
| `betas=["computer-use-2024-10-22"]` | How to unlock the computer-use tool suite |
| `CU_TOOLS` | How to declare `bash_20241022` and `text_editor_20241022` |
| `_run_bash` / `_run_editor` | How to implement local tool executors |
| `run_agent` | The core action loop: request → execute → feed → repeat |
| `ToolUseBlock` | How to read the model's tool requests and match results |
| `stop_reason` | How to detect `end_turn` and break the loop safely |
| System prompts | How to guide and restrict the agent's behaviour |
| Token tracking | How to measure and estimate cost across loop iterations |
| `computer_20241022` | What GUI automation looks like (requires display) |

### Next Steps

- **Example 101** — Multi-agent orchestration: fan out tasks across parallel computer-use agents
- **Production hardening**: add Docker sandboxing, path allowlists, command blocklists
- **GUI agents**: set up Xvfb in Docker and add the `computer_20241022` screenshot loop
- **Custom tools**: extend the dispatcher with web fetch, database query, or API call tools

> **Workshop complete.** Next: example 101 — multi-agent orchestration.